# Modelo A (358+gen_1): grilla de `min_data_in_leaf`

`param_viejo` usa `min_data_in_leaf=4000`, pero nunca se hizo una grilla sobre esta combinación
dataset/pool de variables (358+gen_1) para confirmar que sea el óptimo -- a diferencia de Modelo B, donde
sí se corrió la grilla (`z_grilla_rankeado_lags.ipynb`) y confirmó `min_data_in_leaf=1000` como mejor.

Acá se prueba una grilla de valores alrededor y por debajo de 4000, dejando todo lo demás de
`param_viejo` fijo, con el protocolo estándar (10 semillas, train=202103+202104, corte fijo=12.000).

Nota: se usa `gen_1` **sin rankear** (la versión original), para que el punto `min_data_in_leaf=4000`
reproduzca exactamente la referencia conocida (463.980.000) como chequeo de consistencia. La corrección
de `gen_1` (rankeada) es un experimento aparte -- una vez que tengamos el mejor `min_data_in_leaf` acá,
tiene sentido volver a correr ese punto también con `gen_1` corregida.

### Setup y carga del dataset

In [1]:
rm(list = ls())
gc()

require("data.table")
require("lightgbm")

PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"

t0 <- Sys.time()
dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags_fe_historico_rankeado.csv.gz"))
cat("Dataset cargado:", nrow(dataset), "filas x", ncol(dataset), "columnas en",
    round(as.numeric(Sys.time()-t0, units="secs"),1), "seg\n")

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,653156,34.9,1447540,77.4,1130552,60.4
Vcells,1199219,9.2,8388608,64.0,1928501,14.8


Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



Dataset cargado: 983061 filas x 1622 columnas en 146.4 seg


### Reconstruyo las compuestas y gen_1 (sin rankear, versión original)

In [2]:
agregar_si_existen <- function(dt, nombre_nueva, columnas, fun_combinacion) {
  presentes <- columnas[columnas %in% colnames(dt)]
  if (length(presentes) < 2) return(invisible(NULL))
  dt[, (nombre_nueva) := fun_combinacion(.SD), .SDcols = presentes]
}
suma_filas <- function(sd) Reduce(`+`, sd)

agregar_si_existen(dataset, "consumo_tarjetas_total", c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"), suma_filas)
agregar_si_existen(dataset, "consumototal_tarjetas", c("Visa_mconsumototal", "Master_mconsumototal"), suma_filas)
agregar_si_existen(dataset, "saldo_tarjetas_total", c("Visa_msaldototal", "Master_msaldototal"), suma_filas)
agregar_si_existen(dataset, "limite_tarjetas_total", c("Visa_mlimitecompra", "Master_mlimitecompra"), suma_filas)
agregar_si_existen(dataset, "pagado_tarjetas_total", c("Visa_mpagado", "Master_mpagado"), suma_filas)
agregar_si_existen(dataset, "pagominimo_tarjetas_total", c("Visa_mpagominimo", "Master_mpagominimo"), suma_filas)
agregar_si_existen(dataset, "descuentos_tarjetas_total",
                    c("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos"), suma_filas)

if (all(c("consumototal_tarjetas", "limite_tarjetas_total") %in% colnames(dataset))) {
  dataset[, utilizacion_tarjetas := ifelse(limite_tarjetas_total == 0, 0,
                                            consumototal_tarjetas / limite_tarjetas_total)]
}
if (all(c("Visa_status", "Master_status") %in% colnames(dataset))) {
  dataset[, tarjetas_activas_count := as.integer(Visa_status == 0) + as.integer(Master_status == 0)]
}

agregar_si_existen(dataset, "ingresos_payroll_total", c("mpayroll", "mpayroll2"), suma_filas)
agregar_si_existen(dataset, "prestamos_sin_personales_total",
                    c("mprestamos_prendarios", "mprestamos_hipotecarios"), suma_filas)
agregar_si_existen(dataset, "inversiones_totales",
                    c("minversion1_pesos", "minversion1_dolares", "minversion2"), suma_filas)
agregar_si_existen(dataset, "plazo_fijo_total", c("mplazo_fijo_dolares", "mplazo_fijo_pesos"), suma_filas)

if (all(c("mtransferencias_recibidas", "mtransferencias_emitidas") %in% colnames(dataset))) {
  dataset[, transferencias_netas := mtransferencias_recibidas - mtransferencias_emitidas]
}
agregar_si_existen(dataset, "comisiones_totales_manual", c("mcomisiones_mantenimiento", "mcomisiones_otras"), suma_filas)

dataset[, gen_1 := ctrx_quarter + cpayroll_trx_min3]

cat("Compuestas reconstruidas\n")

Compuestas reconstruidas


### Cargo la lista ganadora (358 vars) y armo train (202103+202104) / future (202106)

In [3]:
campos_base <- fread(paste0(PARAM_carpeta_exp, "campos_canaritos_asesinos_2sd.txt"))$variable
campos_base <- campos_base[campos_base %in% colnames(dataset)]
campos_A <- c(campos_base, "gen_1")
cat("Variables (con gen_1):", length(campos_A), "\n")

dataset_train <- dataset[foto_mes %in% c(202103, 202104)]
dfuture       <- dataset[foto_mes == 202106]

matriz_train  <- data.matrix(dataset_train[, campos_A, with = FALSE])
matriz_future <- data.matrix(dfuture[, campos_A, with = FALSE])

cat("train:", nrow(dataset_train), "filas | future:", nrow(dfuture), "filas\n")

Variables (con gen_1): 359 
train: 326184 filas | future: 164114 filas


### Hiperparámetros base (`param_viejo`) y grilla de `min_data_in_leaf`

In [4]:
param_viejo <- list(
  boosting = "gbdt", objective = "binary", metric = "auc",
  first_metric_only = TRUE, boost_from_average = TRUE, feature_pre_filter = FALSE, verbosity = -100,
  learning_rate = 0.02, feature_fraction = 0.1541961, feature_fraction_bynode = 1,
  num_leaves = 1684, max_depth = -1,
  lambda_l1 = 1.520327, lambda_l2 = 7.548036, min_gain_to_split = 0.2158453, max_delta_step = 0,
  bagging_fraction = 0.8425105, bagging_freq = 6, min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction = 1, neg_bagging_fraction = 1, extra_trees = FALSE, path_smooth = 0,
  cat_l2 = 10, cat_smooth = 10, max_cat_to_onehot = 4, top_rate = 0.2, other_rate = 0.1,
  use_missing = TRUE, is_unbalance = FALSE, num_iterations = 600, early_stopping_rounds = 0,
  max_bin = 31, scale_pos_weight = 0.5, force_row_wise = TRUE
)

SEMILLAS_10 <- c(585581, 249449, 655649, 503039, 798191, 112909, 334667, 421303, 667021, 887923)
CORTE_FIJO <- 12000

GRILLA_MDL <- c(300, 500, 1000, 2000, 3000, 4000, 5000)

### Corro las 10 semillas para cada valor de la grilla

In [5]:
evaluar_mdl <- function(mdl, semillas) {
  ganancias <- c()
  for (semilla in semillas) {
    param_semilla <- param_viejo
    param_semilla$min_data_in_leaf <- mdl
    param_semilla$seed <- semilla

    dtrain <- lgb.Dataset(data = matriz_train, label = dataset_train$clase01, free_raw_data = FALSE)
    modelo <- lgb.train(data = dtrain, param = param_semilla, verbose = -1)
    prediccion <- predict(modelo, matriz_future)

    tbl <- data.table(prob = prediccion, clase_ternaria = dfuture$clase_ternaria)
    setorder(tbl, -prob)
    tbl[, gan_individual := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]
    ganancia <- sum(tbl[1:CORTE_FIJO, gan_individual])

    ganancias <- c(ganancias, ganancia)

    rm(dtrain, modelo)
    gc(verbose = FALSE)
  }
  ganancias
}

resultados <- data.table(min_data_in_leaf = GRILLA_MDL, ganancia_media = NA_real_, desvio = NA_real_)

for (i in seq_along(GRILLA_MDL)) {
  mdl <- GRILLA_MDL[i]
  cat("=== min_data_in_leaf =", mdl, "===\n")
  gan <- evaluar_mdl(mdl, SEMILLAS_10)
  cat("  media:", mean(gan), " desvio:", sd(gan), "\n\n")
  resultados[i, ganancia_media := mean(gan)]
  resultados[i, desvio := sd(gan)]
}

=== min_data_in_leaf = 300 ===
  media: 460020000  desvio: 6153012 

=== min_data_in_leaf = 500 ===
  media: 458370000  desvio: 5082005 

=== min_data_in_leaf = 1000 ===
  media: 461670000  desvio: 4214275 

=== min_data_in_leaf = 2000 ===
  media: 463430000  desvio: 5798668 

=== min_data_in_leaf = 3000 ===
  media: 463650000  desvio: 4674101 

=== min_data_in_leaf = 4000 ===
  media: 463980000  desvio: 5128093 

=== min_data_in_leaf = 5000 ===
  media: 460900000  desvio: 4836666 



### Resumen final

In [6]:
setorder(resultados, -ganancia_media)
print(resultados)

cat("\nReferencia conocida (min_data_in_leaf=4000): 463.980.000\n")
cat("Mejor de la grilla:", resultados[1, min_data_in_leaf], "-> ganancia media:", resultados[1, ganancia_media], "\n")

   min_data_in_leaf ganancia_media  desvio
              <num>          <num>   <num>
1:             4000      463980000 5128093
2:             3000      463650000 4674101
3:             2000      463430000 5798668
4:             1000      461670000 4214275
5:             5000      460900000 4836666
6:              300      460020000 6153012
7:              500      458370000 5082005

Referencia conocida (min_data_in_leaf=4000): 463.980.000
Mejor de la grilla: 4000 -> ganancia media: 463980000 
